# Database Examiner

This module is a test module used to open and examine database table contents to make sure data is being stored and retrieved properly

In [10]:
import os
import pandas as pd
from pprint import pprint

from pleasant_database import DatabaseFile

from backend.utils.file_utils import EDirectories
from backend.database_modules.managers.transaction_manager import TransactionsTableManager, UpdatesTableManager
from backend.database_modules.managers.summary_manager import SummariesTableManager
from backend.database_modules.managers.budget_manager import BudgetsTableManager

In [11]:
database_file = DatabaseFile(EDirectories.DB_FILENAME, EDirectories.DB_DIR)

TEST_DB_DIR = os.path.join(os.getcwd(), "backend", "tests", "test_databases")
TEST_DB_FILENAME = "test_database.db"
test_database_file = DatabaseFile(TEST_DB_FILENAME, TEST_DB_DIR)

In [12]:
live_updates_manager = UpdatesTableManager(database_file)
live_transactions_manager = TransactionsTableManager(database_file, live_updates_manager)
live_summaries_manager = SummariesTableManager(database_file)
live_budgets_manager = BudgetsTableManager(database_file)

# test_updates_manager = UpdatesTableManager(test_database_file)
# test_transactions_manager = TransactionsTableManager(test_database_file)
# test_summaries_manager = SummariesTableManager(test_database_file)
# test_budgets_manager = BudgetsTableManager(test_database_file)

2026-04-02 17:54:06,231 - INFO - pleasant_database.database_connections - Creating connection engine to /Users/eliasrodkey/git_repo/budgy_2.0/data/databases/budgy_financial_transaction.db...
2026-04-02 17:54:06,246 - INFO - pleasant_database.database_connections - Creating session for Engine(sqlite:////Users/eliasrodkey/git_repo/budgy_2.0/data/databases/budgy_financial_transaction.db)...
2026-04-02 17:54:06,246 - INFO - pleasant_database.database_manager - DatabaseManager session started with table: transaction_updates
2026-04-02 17:54:06,256 - INFO - pleasant_database.database_connections - Creating connection engine to /Users/eliasrodkey/git_repo/budgy_2.0/data/databases/budgy_financial_transaction.db...
2026-04-02 17:54:06,256 - INFO - pleasant_database.database_connections - Creating session for Engine(sqlite:////Users/eliasrodkey/git_repo/budgy_2.0/data/databases/budgy_financial_transaction.db)...
2026-04-02 17:54:06,256 - INFO - pleasant_database.database_manager - DatabaseManage

In [13]:
live_transactions_manager.upload_all_csvs()

2026-04-02 17:54:41,747 - INFO - backend.database_modules.managers.transaction_manager - Beginning upload of all csv files in /Users/eliasrodkey/git_repo/budgy_2.0/data/csv_downloads to transactions
2026-04-02 17:54:41,766 - WARNING - pleasant_database.database_manager - No items found in database by using attributes.
2026-04-02 17:54:41,766 - INFO - backend.database_modules.managers.transaction_manager - CSV file SoFi-Relay-All-Transactions_2026-03-13.csv has not yet been uploaded to the database.
2026-04-02 17:54:41,767 - INFO - backend.database_modules.managers.transaction_manager - Beginning upload of CSV file to database: SoFi-Relay-All-Transactions_2026-03-13.csv
2026-04-02 17:54:41,780 - INFO - backend.csv_modules.transactions_csv_loader - Iterating and validating CSV file: SoFi-Relay-All-Transactions_2026-03-13.csv
2026-04-02 17:54:41,785 - INFO - backend.database_modules.managers.transaction_manager - Completed upload of CSV file to database: SoFi-Relay-All-Transactions_2026-0

In [16]:
live_transactions_manager.to_dataframe().info()
live_updates_manager.to_dataframe().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1171 entries, 0 to 1170
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 1171 non-null   int64         
 1   authorized_date    1171 non-null   datetime64[ns]
 2   posted_date        1171 non-null   datetime64[ns]
 3   status             1171 non-null   object        
 4   account_name       1171 non-null   object        
 5   description        1171 non-null   object        
 6   primary_category   1171 non-null   object        
 7   detailed_category  1171 non-null   object        
 8   amount             1171 non-null   float64       
 9   repayment          1171 non-null   bool          
 10  exclude            1171 non-null   bool          
 11  base_hash          1171 non-null   object        
 12  uq_hash            1171 non-null   object        
dtypes: bool(2), datetime64[ns](2), float64(1), int64(1), object(7)


# Categories to Avoid

* We can safely ignore **account transfers**, although I need to go through and make sure there aren't any hidden transactions in that category, venmo's tend to slip in here pretty often.
* **credit card payments** is another one that we can leave out entirely, it is paying off my credit cards. Maybe I can use this to make sure my balance is zero each month in the future?
* **investment transfers** is moving from one of my accounts to another, could be good to track what percent of takehome I am putting towards personal investments but can exclude from spending calculation
* **savings transfers** can be safely excluded